In [ ]:
med_hb_raw = session.table(f"{DB}.{RAW}.STREAM_T_MEDICATION_HEALTH_BEHAVIOR").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
ind_cols = ["THRIVE_FAILURE_IND", "DATE_UNKNOWN"]
code_cols = ["PRESCRIBED_FREQUENCY_CODE", "ADMINISTER_METHOD_CODE", "CONSENT_DECISION_CODE"]
desc_cols = ["MEDICATION_CMNT", "DOSAGE_DESC", "CLN_RESP_ADVERSE_EFFECT_CMNT", "CONSENT_COMMENTS"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["SOURCE_FILE_NAME"]

all_cols = [c.name for c in med_hb_raw.schema.fields]
select_exprs = []
for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in desc_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("NA")).otherwise(trim(col(c))), lit("NA")).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

med_hb_clean = med_hb_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_med_hb = med_hb_clean.filter(
    col("MHP_ID").is_not_null() &
    col("MED_MED_ID").is_not_null() &
    col("PERSON_PERSON_PATIENT_ID").is_not_null()
)

invalid_med_hb = med_hb_clean.filter(
    col("MHP_ID").is_null() | col("MED_MED_ID").is_null() | col("PERSON_PERSON_PATIENT_ID").is_null()
).with_column(
    "ERROR_REASON",
    when(col("MHP_ID").is_null(), lit("MHP_ID_NULL"))
    .when(col("MED_MED_ID").is_null(), lit("MED_MED_ID_NULL"))
    .otherwise(lit("PERSON_PERSON_PATIENT_ID_NULL"))
)
print("Valid/invalid split defined")

In [ ]:
checksum_cols = [
    ("MHP_ID", "number"), ("MED_MED_ID", "number"), ("HPP_HPP_ID", "number"),
    ("PERSON_PERSON_PATIENT_ID", "number"), ("PERSON_PERSON_PRESCRIBER_ID", "number"),
    ("PRESCRIBER_PCS_PCS_ID", "number"), ("CONSENT_BY_PERSON_PERSON_ID", "number"),
    ("ADD_PERSON_ID", "number"), ("ADD_ORGN_ID", "number"),
    ("MOD_PERSON_ID", "number"), ("MOD_ORGN_ID", "number"),
    ("THRIVE_FAILURE_IND", "string"), ("DATE_UNKNOWN", "string"),
    ("PRESCRIBED_FREQUENCY_CODE", "string"), ("ADMINISTER_METHOD_CODE", "string"),
    ("CONSENT_DECISION_CODE", "string"), ("MEDICATION_CMNT", "string"),
    ("DOSAGE_DESC", "string"), ("CLN_RESP_ADVERSE_EFFECT_CMNT", "string"),
    ("CONSENT_COMMENTS", "string"), ("ADD_USER", "string"), ("MOD_USER", "string"),
    ("START_DATE", "timestamp"), ("END_DATE", "timestamp"),
    ("LAST_REFILL_DATE", "timestamp"), ("CONSENT_DATE", "timestamp")
]
checksum_exprs = []
for col_name, col_type in checksum_cols:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

med_hb_final = valid_med_hb.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

med_hb_final.write.save_as_table(f"{DB}.{STG}.{STG_MEDICATION_HEALTH_BEHAVIOR}", mode="truncate")
print(f"MED_HB loaded into {STG}.{STG_MEDICATION_HEALTH_BEHAVIOR}")

In [ ]:
invalid_count = invalid_med_hb.count()
if invalid_count > 0:
    invalid_med_hb.select(
        lit("T_MEDICATION_HEALTH_BEHAVIOR").alias("TABLE_NAME"), col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"), col("RAW_LOAD_TIMESTAMP")
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_MEDICATION_HEALTH_BEHAVIOR}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_MEDICATION_HEALTH_BEHAVIOR_LOAD', 'STAGING',
    datetime.now(), datetime.now(), rows_processed, invalid_count, status, None)
session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_MEDICATION_HEALTH_BEHAVIOR_LOAD', 'STAGING',
    f'MHB staging completed. Rows: {rows_processed}, Failed: {invalid_count}')
print(f"STG_MHB complete | Valid: {rows_processed} | Invalid: {invalid_count}")